# SFSC 2026 — V5 Step 4: aspect-matched refiner, fold-0 pilot (Colab / NVIDIA L4)

Step 4 of `v5/V5_PLAN_R2.md`. Each of V4's candidates is cut out in a frame rotated so its long axis is horizontal
(384 x 128 network input, 50% padding along the axis) and redrawn by a crop UNet trained on folds 1-3.

| | |
|---|---|
| inputs (5 ch) | raw, flattened, V3 probability, this candidate's mask, all other candidates' mask |
| target | P(marked \| this filament drawn) = mean over the annotator sets that drew it |
| loss | 0.5 BCE + 1.0 Lovász hinge (V3's recipe; the step-2 pilot showed Lovász matters) |
| mask | probability >= t*, t* = J*/(1+J*) (fixed point of the expected IoU), largest piece touching the seed |
| decision | V4's cross-fitted keep decisions (fusion run), painted disjoint in score order: only the masks change |

Gate on fold 0 (177 images): PQ +0.003 and better in 3 of 4 date splits.
Output: `MyDrive/SFSC2026/outputs/sfsc_v5/s4_refiner/`.

**Run:** Runtime -> L4 GPU -> Run all. Needs the fusion run `outputs/sfsc_v5/fusion_sfsc_unet_v3_aug` (or `_sfsc_unet_v3`) on Drive.


## Cell 01 — Drive access, inputs (read only), output folder


In [ ]:
from pathlib import Path
import os, shutil, subprocess

IN_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG"))
DRIVE_MODE = os.environ.get("SFSC_DRIVE_MODE", "mount" if IN_COLAB else "local")   # mount | rclone | local
RCLONE_REMOTE = os.environ.get("SFSC_RCLONE_REMOTE", "gdrive")
DEBUG = os.environ.get("SFSC_DEBUG", "0") == "1"
OUT_NAME = "s4_refiner_debug" if DEBUG else "s4_refiner"
MAP_RUNS, AUG, TAG = ["sfsc_unet_v3"], False, "sfsc_unet_v3"          # V4's maps exactly
FUSION_RUNS = ("fusion_sfsc_unet_v3_aug", "fusion_sfsc_unet_v3")      # keep decisions (first one found)
LOCAL_ROOT = Path(os.environ.get("SFSC_LOCAL_ROOT", "/content/sfsc_v5_s4" if IN_COLAB else "/tmp/sfsc_v5_s4"))
LOCAL_DATA = LOCAL_ROOT / "dataset"
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

def rclone(*args, check=True):
    return subprocess.run(["rclone", *args, "--transfers", "16", "--checkers", "16"], check=check)

if DRIVE_MODE == "mount":
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/SFSC2026")
elif DRIVE_MODE == "rclone":
    if shutil.which("rclone") is None:
        subprocess.run("curl -fsSL https://rclone.org/install.sh | bash", shell=True, check=True)
    DRIVE_ROOT = LOCAL_ROOT / "drive_mirror"
    remote = f"{RCLONE_REMOTE}:SFSC2026"
    if not os.environ.get("SFSC_DATASET"):
        rclone("copy", f"{remote}/data", str(DRIVE_ROOT / "data"), "--include", "/sfsc2026-data.zip")
    rclone("copy", f"{remote}/outputs/sfsc_unet_v3", str(DRIVE_ROOT / "outputs" / "sfsc_unet_v3"),
           "--include", "/{folds_v3.csv,frozen_recipe_v3.json}", "--include", "/fold*/{config,done}.json",
           "--include", "/oof/**", "--include", "/test_probs/**")
    rclone("copy", f"{remote}/outputs/sfsc_fusion_v4/detections", str(DRIVE_ROOT / "outputs" / "sfsc_fusion_v4" / "detections"))
    rclone("copy", f"{remote}/outputs/sfsc_v5/{OUT_NAME}", str(DRIVE_ROOT / "outputs" / "sfsc_v5" / OUT_NAME),
           "--include", "/fold*/{best.pt,done.json,history.csv}", check=False)          # a finished refiner is reused
else:
    DRIVE_ROOT = Path(os.environ["SFSC_DRIVE_ROOT"])

DATA_ZIP = DRIVE_ROOT / "data" / "sfsc2026-data.zip"
DRIVE_EXTRACTED = DRIVE_ROOT / "data" / "SFSCDataset"
V3_RUN = DRIVE_ROOT / "outputs" / "sfsc_unet_v3"
MAP_DIRS = [V3_RUN]
DET_DIR = DRIVE_ROOT / "outputs" / "sfsc_fusion_v4" / "detections"
V4_RUN_BASE = DRIVE_ROOT / "outputs" / "sfsc_v5" / OUT_NAME                 # this run's folder (name kept for V4's code)

def fetch_fusion():
    """The fusion run's cross-fitted keep scores (written by the CPU fusion notebook; fetched when needed)."""
    for r in FUSION_RUNS:
        d = DRIVE_ROOT / "outputs" / "sfsc_v5" / r
        if DRIVE_MODE == "rclone":
            rclone("copy", f"{RCLONE_REMOTE}:SFSC2026/outputs/sfsc_v5/{r}", str(d), "--include", "/{image_stats.npz,selection_v5.json}",
                   check=False)
        if (d / "image_stats.npz").is_file():
            return d
    raise FileNotFoundError(f"No fusion run with image_stats.npz among {FUSION_RUNS}")

def push(sub=""):
    if DRIVE_MODE == "rclone":
        rclone("copy", str(RUN / sub), f"{RCLONE_REMOTE}:SFSC2026/outputs/sfsc_v5/{OUT_NAME}/{sub}".rstrip("/"), check=False)

print("Drive mode:", DRIVE_MODE, "| output:", V4_RUN_BASE, "| debug:", DEBUG)


## Cell 02 — Dependencies


In [ ]:
import importlib.util, sys
REQUIRED = {"segmentation_models_pytorch": "segmentation-models-pytorch>=0.3.4", "timm": "timm>=0.9",
            "sklearn": "scikit-learn>=1.3", "pycocotools": "pycocotools>=2.0.7", "cv2": "opencv-python-headless>=4.8",
            "scipy": "scipy>=1.10", "pandas": "pandas>=2.0", "tqdm": "tqdm>=4.66"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
import torch, segmentation_models_pytorch as smp
print("torch", torch.__version__, "| smp", smp.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


## Cell 03 — Imports, configuration (edit here), settings read from V3, helpers


In [ ]:
import copy, gc, hashlib, json, math, multiprocessing as mp, pickle, platform
import random, re, shutil, subprocess, sys, tempfile, time, warnings, zipfile
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from dataclasses import dataclass, asdict
from functools import lru_cache

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from pycocotools import mask as mask_api
from sklearn.ensemble import HistGradientBoostingRegressor
from tqdm.auto import tqdm

@dataclass
class Config:
    # ---- run control ----
    debug_run: bool = False                  # True = quick pipeline check (separate *_debug folder)
    local_dataset_override: str | None = os.environ.get("SFSC_DATASET") or None
    export_test: bool = True
    skip_empty_records: bool = True          # the organizers' scorer never sees annotator sets without filaments

    # ---- detector: the second model family ----
    det_weights: str = "yolo11m-seg.pt"
    det_imgsz: int = 1280
    det_epochs: int = 40
    det_batch: int = 4
    det_cls_gain: float = 1.5                # classification-loss weight (Ultralytics default 0.5)
    det_workers: int = 8
    det_save_every: int = 2                  # copy the resume checkpoint to Drive every N epochs
    det_folds: tuple = (0, 1, 2, 3)
    det_conf_floor: float = 0.05             # lowest detection score kept as evidence
    det_nms_iou: float = 0.6
    det_max_det: int = 200

    # ---- fusion ----
    det_min_area: int = 32                   # px, smaller detections are ignored
    det_link_frac: float = 0.2               # a detection supports a candidate if they share this fraction of the smaller one
    det_only_floor: float = 0.10             # lowest score for a detection to create a new candidate
    det_only_max_claimed: float = 0.30       # ...and at most this fraction of it may already be covered
    det_grow: int = 4                        # px the detection is grown before UNet pixels are claimed inside it
    det_claim_t: float = 0.20                # UNet probability needed for a claimed pixel
    det_claim_min_frac: float = 0.50         # use UNet pixels if they cover this fraction of the detection, else its own mask
    det_only_min_area: int = 64

    # ---- selection ----
    min_gain: float = 0.003                  # a V4 variant must beat re-scored V3 by this out-of-fold PQ...
    min_fold_wins: int = 3                   # ...and win in at least this many folds
    bootstrap: int = 2000
    cpu_pool_workers: int | None = None
    aug_levels: tuple = (0.15, 0.20, 0.25, 0.30, 0.50, 0.60, 0.70)   # step 1: extra t_low levels (V4: 0.40)
    run_diagnostics: bool = False
    pilot_fold: int = int(os.environ.get("SFSC_PILOT_FOLD", "0"))
    match_min_iou: float = 0.1               # an annotated filament belongs to a candidate's target from this IoU
    crop_w: int = 384                         # network input along the long axis
    crop_h: int = 128                         # ... across it
    pad_along: float = 1.5                    # frame length = 1.5 x the candidate's extent along its axis
    min_len: int = 96                         # native px
    across_margin: int = 24                   # native px on each side of the candidate's extent across its axis
    gt_seed_fraction: float = 0.3             # training: jittered annotated filaments instead of real candidates
    encoder: str = "tu-tf_efficientnetv2_s.in21k_ft_in1k"
    epochs: int = 24
    samples_per_epoch: int = 16000
    batch: int = 64
    lr: float = 4e-4
    weight_decay: float = 1e-4
    workers: int = 12

CFG = Config()
if DEBUG:
    CFG.epochs, CFG.samples_per_epoch = 2, 1024
if CFG.debug_run:
    CFG.det_folds, CFG.det_epochs = (0,), 2
DEBUG_LIMIT = 24 if CFG.debug_run else None

# ---- settings that must match the V3 run are read from it, not retyped ----
_cfg_files = sorted(V3_RUN.glob("fold*/config.json"))
if not _cfg_files:
    raise FileNotFoundError(f"No fold*/config.json under {V3_RUN}")
V3CFG = json.loads(_cfg_files[0].read_text())["config"]
for _k in ("seed", "n_folds", "disk_radius_fraction", "radial_bins", "flat_median_kernel", "flat_std", "contact_radius",
           "connectivity", "fill_holes", "min_core_area", "contact_assign_radius", "scorer_prefilter_max",
           "scorer_prefilter_area"):
    setattr(CFG, _k, V3CFG[_k])
CFG.expected_shape = tuple(int(x) for x in V3CFG["expected_shape"])
CFG.scorer_taus = tuple(float(t) for t in V3CFG["scorer_taus"])

RUN = V4_RUN_BASE
RUN.mkdir(parents=True, exist_ok=True)
random.seed(CFG.seed); np.random.seed(CFG.seed)
cv2.setNumThreads(0)

CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 2)
CPU_POOL = max(2, CPU_COUNT - 2) if CFG.cpu_pool_workers is None else CFG.cpu_pool_workers

def hash_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(1 << 20), b""):
            h.update(part)
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode()).hexdigest()

def write_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp = tempfile.mkstemp(prefix=".json_", dir=str(path.parent))
    try:
        with os.fdopen(fd, "w") as f:
            json.dump(obj, f, indent=2, default=str)
        os.replace(temp, path)
    finally:
        Path(temp).unlink(missing_ok=True)

def save_npy_atomic(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp = tempfile.mkstemp(prefix=".npy_", dir=str(path.parent))
    try:
        with os.fdopen(fd, "wb") as f:
            np.save(f, value, allow_pickle=False)
        os.replace(temp, path)
    finally:
        Path(temp).unlink(missing_ok=True)

def copy_atomic(src, dst):
    """Copy unless a file of the same size is already there."""
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_file() and dst.stat().st_size == src.stat().st_size:
        return
    copy_force(src, dst)

def copy_force(src, dst):
    """Always copy (checkpoints of the same size differ in content)."""
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    temp = dst.with_suffix(dst.suffix + ".part")
    shutil.copyfile(src, temp)
    os.replace(temp, dst)

def save_pickle(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".part")
    with open(temp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temp, path)

print("CPU cores:", CPU_COUNT, "| CPU scoring pool:", CPU_POOL)
print("Run folder:", RUN, "| debug_run =", CFG.debug_run)
print("From V3: seed", CFG.seed, "| image", CFG.expected_shape, "| disk radius fraction", CFG.disk_radius_fraction)


## Cell 04 — (fusion notebook / V4, unchanged)


In [ ]:
ANNOTATION_BASENAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
LOCAL_ZIP = LOCAL_ROOT / "sfsc_dataset.zip"

def locate_dataset(root):
    root = Path(root)
    hits = sorted(root.rglob(ANNOTATION_BASENAME)) if root.is_dir() else []
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected one {ANNOTATION_BASENAME} under {root}; found {len(hits)}")
    ann = hits[0]
    train_dir = ann.parent / "train_images"
    test_dir = ann.parent.parent / "test" / "test_images"
    if not (train_dir.is_dir() and test_dir.is_dir()):
        raise FileNotFoundError(f"Missing train_images or test_images near {ann}")
    return ann, train_dir, test_dir

def image_paths(folder):
    return {p.name: p for p in Path(folder).iterdir()
            if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg")}

def staged_and_ready(root):
    try:
        a, t, s = locate_dataset(root)
        return a.stat().st_size > 1000 and len(image_paths(t)) > 0 and len(image_paths(s)) > 0
    except (FileNotFoundError, OSError):
        return False

def safe_extract(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    resolved_root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in tqdm(archive.infolist(), desc="Extract dataset"):
            if member.is_dir():
                continue
            dest = (destination / member.filename).resolve()
            if not dest.is_relative_to(resolved_root):
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
            if dest.is_file() and dest.stat().st_size == member.file_size:
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            archive.extract(member, destination)

if CFG.local_dataset_override:
    DATA_ROOT = Path(CFG.local_dataset_override)
    if not staged_and_ready(DATA_ROOT):
        raise FileNotFoundError(f"Dataset override not ready: {DATA_ROOT}")
elif staged_and_ready(LOCAL_DATA):
    DATA_ROOT = LOCAL_DATA
else:
    DATA_ROOT = LOCAL_DATA
    if DATA_ZIP.is_file():
        print("Copying ZIP to local SSD once per runtime...")
        copy_atomic(DATA_ZIP, LOCAL_ZIP)
        safe_extract(LOCAL_ZIP, DATA_ROOT)
    elif DRIVE_EXTRACTED.is_dir():
        print("Copying extracted dataset folder from Drive to local SSD...")
        shutil.copytree(DRIVE_EXTRACTED, DATA_ROOT, dirs_exist_ok=True)
    else:
        raise FileNotFoundError(f"Expected {DATA_ZIP} or {DRIVE_EXTRACTED}. Or set CFG.local_dataset_override.")
if not staged_and_ready(DATA_ROOT):
    raise RuntimeError("Local staging incomplete. Check ZIP contents and free disk space.")

ANN_PATH, TRAIN_DIR, TEST_DIR = locate_dataset(DATA_ROOT)
TRAIN_PATHS, TEST_PATHS = image_paths(TRAIN_DIR), image_paths(TEST_DIR)
for label, paths in (("train", TRAIN_PATHS), ("test", TEST_PATHS)):
    if len({Path(n).stem for n in paths}) != len(paths):
        raise ValueError(f"Duplicate {label} image stems; IDs would collide.")

def read_gray(path):
    with Image.open(path) as im:
        if im.mode == "L":
            return np.asarray(im, dtype=np.uint8).copy()
        if im.mode != "RGB":
            raise ValueError(f"Unexpected image mode {im.mode}: {path}")
        arr = np.asarray(im, dtype=np.uint8)
        d1 = np.abs(arr[:, :, 0].astype(np.int16) - arr[:, :, 1])
        d2 = np.abs(arr[:, :, 0].astype(np.int16) - arr[:, :, 2])
        if d1.max() > 2 or d2.max() > 2:
            raise ValueError(f"Non-grayscale RGB image: {path}")
        return arr[:, :, 0].copy()

RAW = json.loads(ANN_PATH.read_text())
ANN_SHA = hash_file(ANN_PATH)
print("Annotation JSON:", ANN_PATH, "| sha256", ANN_SHA[:16])
print("JPEGs:", len(TRAIN_PATHS), "train /", len(TEST_PATHS), "test | free local GiB:",
      round(shutil.disk_usage(LOCAL_ROOT).free / 2**30, 1))


## Cell 05 — (fusion notebook / V4, unchanged)


In [ ]:
H, W = CFG.expected_shape

def native_rle(rle):
    counts = rle["counts"]
    if isinstance(counts, str):
        counts = counts.encode("ascii")
    return {"size": [int(x) for x in rle["size"]], "counts": counts}

def json_rle(rle):
    counts = rle["counts"]
    return {"size": [int(x) for x in rle["size"]],
            "counts": counts.decode("ascii") if isinstance(counts, bytes) else counts}

def decode_mask(rle):
    a = mask_api.decode(native_rle(rle))
    return np.any(a, axis=2) if a.ndim == 3 else (a > 0)

def segmentation_to_rle(seg, h, w):
    if isinstance(seg, dict):
        if list(seg.get("size", [h, w])) != [h, w]:
            raise ValueError("Annotation RLE shape mismatch")
        rle = (mask_api.frPyObjects(seg, h, w) if isinstance(seg.get("counts"), list)
               else native_rle({"size": [h, w], "counts": seg["counts"]}))
    elif isinstance(seg, list) and len(seg):
        polys = seg if isinstance(seg[0], (list, tuple)) else [seg]
        if any(len(p) < 6 or len(p) % 2 for p in polys):
            raise ValueError("Malformed polygon segmentation")
        rle = mask_api.merge(mask_api.frPyObjects(polys, h, w))
    else:
        raise ValueError("Unsupported empty/malformed segmentation")
    return json_rle(rle)

STEMS = defaultdict(list)
for _n in TRAIN_PATHS:
    STEMS[Path(_n).stem].append(_n)

def canonical_filename(s):
    name = Path(str(s)).name
    if name in TRAIN_PATHS:
        return name
    cands = STEMS.get(Path(name).stem, [])
    if len(cands) == 1:
        return cands[0]
    raise ValueError(f"Unresolvable annotated JPEG: {s}")

IMAGES_BY_ID = {}
for im in RAW["images"]:
    rid = str(im["id"])
    if rid in IMAGES_BY_ID:
        raise ValueError(f"Duplicate image record {rid}")
    if (int(im["height"]), int(im["width"])) != (H, W):
        raise ValueError(f"Unexpected dimensions for record {rid}")
    IMAGES_BY_ID[rid] = im

ANNOTATIONS_BY_ID = defaultdict(list)
_seen = set()
for a in RAW["annotations"]:
    aid, rid = str(a["id"]), str(a["image_id"])
    if aid in _seen or rid not in IMAGES_BY_ID:
        raise ValueError(f"Duplicate annotation or missing image: {aid} / {rid}")
    if int(a.get("iscrowd", 0)):
        raise ValueError(f"Unexpected crowd annotation {aid}")
    if int(a.get("category_id", 1)) not in (1, 2, 3, 4):
        raise ValueError(f"Unexpected category {a.get('category_id')} in {aid}")
    _seen.add(aid)
    ANNOTATIONS_BY_ID[rid].append(a)

RECORD_IDS_BY_FILE = defaultdict(list)
for rid, im in IMAGES_BY_ID.items():
    RECORD_IDS_BY_FILE[canonical_filename(im["file_name"])].append(rid)
for k in RECORD_IDS_BY_FILE:
    RECORD_IDS_BY_FILE[k].sort()
UNIQUE_FILES = sorted(RECORD_IDS_BY_FILE)

FILE_HASHES = {}
with ThreadPoolExecutor(max_workers=8) as pool:
    for name, digest in zip(UNIQUE_FILES, pool.map(lambda n: hash_file(TRAIN_PATHS[n]), UNIQUE_FILES)):
        FILE_HASHES[name] = digest

unannotated = sorted(set(TRAIN_PATHS) - set(UNIQUE_FILES))
if unannotated:
    warnings.warn(f"{len(unannotated)} train JPEGs have no annotation record; they are not used.")
print("Annotated JPEGs:", len(UNIQUE_FILES), "| annotator records:", len(IMAGES_BY_ID),
      "| instances:", len(RAW["annotations"]))
print("Records per JPEG:", pd.Series([len(v) for v in RECORD_IDS_BY_FILE.values()]).value_counts().sort_index().to_dict())


## Cell 06 — (fusion notebook / V4, unchanged)


In [ ]:
CY, CX = (H - 1) / 2.0, (W - 1) / 2.0
DISK_R = CFG.disk_radius_fraction * min(H, W) / 2.0
_YY, _XX = np.mgrid[0:H, 0:W]
RADIAL = (np.hypot(_YY - CY, _XX - CX) / DISK_R).astype(np.float32)
YY_FLAT = _YY.ravel().astype(np.float64)
XX_FLAT = _XX.ravel().astype(np.float64)
del _YY, _XX
DISK = RADIAL < 1.0
RBIN = np.minimum((RADIAL * CFG.radial_bins).astype(np.int32), CFG.radial_bins - 1)
RBIN_DISK = RBIN[DISK]

IMG_MEAN, IMG_STD = 0.5, 0.25
RAD_MEAN, RAD_STD, RAD_CLIP = 0.6, 0.35, 1.25

def flatten_disk(img):
    """Divide by the radial limb-darkening profile, then by a large local median; off-disk = neutral (1.0 -> 128)."""
    x = img.astype(np.float32)
    nb = CFG.radial_bins
    s = np.bincount(RBIN_DISK, weights=x[DISK], minlength=nb)
    c = np.bincount(RBIN_DISK, minlength=nb)
    valid = c > 0
    prof = np.interp(np.arange(nb), np.flatnonzero(valid), (s[valid] / c[valid]))
    k = 7
    prof = np.convolve(np.pad(prof, k // 2, mode="edge"), np.ones(k) / k, mode="valid")
    ratio = x / np.maximum(prof[RBIN], 1.0).astype(np.float32)
    ratio[~DISK] = 1.0
    small = cv2.resize(ratio, (W // 4, H // 4), interpolation=cv2.INTER_AREA)
    small_u8 = np.clip(small * 128.0, 0, 255).astype(np.uint8)
    ksize = max(3, int(round(CFG.flat_median_kernel * H / 2048)) | 1)
    med = cv2.medianBlur(small_u8, ksize).astype(np.float32) / 128.0
    med = cv2.resize(med, (W, H), interpolation=cv2.INTER_LINEAR)
    flat = ratio / np.maximum(med, 0.25)
    flat[~DISK] = 1.0
    return np.clip(flat * 128.0, 0, 255).astype(np.uint8)

CACHE = LOCAL_ROOT / "cache" / f"v3_{ANN_SHA[:12]}_{H}x{W}_r{CFG.disk_radius_fraction:.4f}_k{CFG.flat_median_kernel}_c{CFG.contact_radius}"
for sub in ("img", "flat", "soft", "contact", "gt"):
    (CACHE / sub).mkdir(parents=True, exist_ok=True)

def cache_key(split, name):
    return f"{split}__{Path(name).stem}"

def cache_image(split, name, path):
    key = cache_key(split, name)
    ip, fp = CACHE / "img" / f"{key}.npy", CACHE / "flat" / f"{key}.npy"
    if ip.is_file() and fp.is_file():
        return
    img = read_gray(path)
    if img.shape != (H, W):
        raise ValueError(f"Wrong JPEG shape {img.shape}: {path}")
    save_npy_atomic(ip, img)
    save_npy_atomic(fp, flatten_disk(img))
def build_gt(name):
    """Flat pixel indices and areas of every annotated filament, per annotator set."""
    gp = CACHE / "gt" / f"{cache_key('train', name)}_v4.pkl"
    if gp.is_file():
        with open(gp, "rb") as f:
            return pickle.load(f)
    records = []
    for rid in RECORD_IDS_BY_FILE[name]:
        idx_list, areas = [], []
        for ann in ANNOTATIONS_BY_ID[rid]:
            m = decode_mask(segmentation_to_rle(ann["segmentation"], H, W))
            if not m.any():
                raise ValueError(f"Empty filament annotation {ann['id']}")
            flat_idx = np.flatnonzero(m).astype(np.int32)
            idx_list.append(flat_idx); areas.append(len(flat_idx))
        records.append({"id": rid, "idx": idx_list, "area": np.asarray(areas, np.float64)})
    gt = {"name": name, "records": records}
    save_pickle(gp, gt)
    return gt

with ThreadPoolExecutor(max_workers=min(8, CPU_COUNT)) as pool:
    list(tqdm(pool.map(lambda n: cache_image("train", n, TRAIN_PATHS[n]), UNIQUE_FILES),
              total=len(UNIQUE_FILES), desc="Cache train images + flattened"))
    list(tqdm(pool.map(lambda n: cache_image("test", n, TEST_PATHS[n]), sorted(TEST_PATHS)),
              total=len(TEST_PATHS), desc="Cache test images + flattened"))
    _gt_list = list(tqdm(pool.map(build_gt, UNIQUE_FILES), total=len(UNIQUE_FILES), desc="Ground-truth index"))

GT_INDEX, N_EMPTY_RECORDS = {}, 0
for g in _gt_list:
    scored = [r for r in g["records"] if len(r["idx"])] if CFG.skip_empty_records else list(g["records"])
    N_EMPTY_RECORDS += len(g["records"]) - len(scored)
    GT_INDEX[g["name"]] = {"name": g["name"], "records": scored, "n_all": len(g["records"])}
del _gt_list

@lru_cache(maxsize=64)
def cached_arrays(key, with_targets=True):
    """(image, flattened image) as memory maps. The second argument is kept for compatibility with V3's functions."""
    return tuple(np.load(CACHE / s / f"{key}.npy", mmap_mode="r") for s in ("img", "flat"))

print("Scored annotator sets:", sum(len(g["records"]) for g in GT_INDEX.values()),
      "| filaments:", sum(len(r["idx"]) for g in GT_INDEX.values() for r in g["records"]),
      "| annotator sets without filaments (skipped: %s): %d" % (CFG.skip_empty_records, N_EMPTY_RECORDS))


## Cell 07 — (fusion notebook / V4, unchanged)


In [ ]:
FOLDS = pd.read_csv(V3_RUN / "folds_v3.csv", dtype={"file_name": str, "group": str, "v1_split": str})
if set(FOLDS.file_name) != set(UNIQUE_FILES):
    raise ValueError("V3's folds_v3.csv does not match this dataset.")
FOLDS["fold"] = FOLDS["fold"].astype(int)
FOLD_OF = dict(zip(FOLDS.file_name, FOLDS.fold))

V3_FROZEN = json.loads((V3_RUN / "frozen_recipe_v3.json").read_text())
V3_REC, V3_SHA8 = V3_FROZEN["recipe"], V3_FROZEN["sha256"][:8]
_g = V3_REC["geometry"]
GEOM = (float(_g["t_low"]), None if _g["contact"] is None else float(_g["contact"]), int(_g["offset"]))
V3_TAU = None if V3_REC.get("scorer_tau") is None else float(V3_REC["scorer_tau"])
V3_FOLDS = sorted(int(f) for f in V3_REC["folds"])
OOF_TAG = object_hash({"tta": list(V3CFG["tta_final"]), "tile": V3CFG["tile"], "stride": V3CFG["stride"]})[:10]
MAPS_LOCAL = LOCAL_ROOT / "v3_maps"

def stage_npz(task):
    """Drive .npz (fg, ct) -> two local .npy files that CPU workers can memory-map."""
    src, lf, lc = task
    if not (lf.is_file() and lc.is_file()):
        with np.load(src) as z:
            fg, ct = z["fg"], z["ct"]
        save_npy_atomic(lf, fg); save_npy_atomic(lc, ct)
    return str(lf), str(lc)

def find_oof_dir(fold):
    sha = json.loads((V3_RUN / f"fold{fold}" / "done.json").read_text())["best_sha256"][:10]
    exact = V3_RUN / "oof" / f"fold{fold}_{sha}_{OOF_TAG}"
    if exact.is_dir():
        return exact
    cands = sorted((d for d in (V3_RUN / "oof").glob(f"fold{fold}_*") if d.is_dir()),
                   key=lambda d: len(list(d.glob("*.npz"))))
    if not cands:
        raise FileNotFoundError(f"No out-of-fold maps for fold {fold} under {V3_RUN / 'oof'}. Run V3's Cell 19.")
    return cands[-1]


def stage_avg(task):
    """One or more Drive .npz maps (fg, ct uint8) -> their average as two local .npy files."""
    srcs, lf, lc = task
    if not (lf.is_file() and lc.is_file()):
        fgs, cts = [], []
        for s in srcs:
            with np.load(s) as z:
                fgs.append(z["fg"].astype(np.float32)); cts.append(z["ct"].astype(np.float32))
        fg = np.rint(np.mean(fgs, 0)).astype(np.uint8); ct = np.rint(np.mean(cts, 0)).astype(np.uint8)
        save_npy_atomic(lf, fg); save_npy_atomic(lc, ct)
    return str(lf), str(lc)

def oof_dir_of(run_dir, fold):
    """V3: the directory its own done.json points to; other runs: the fold directory with the most maps."""
    if run_dir == V3_RUN:
        return find_oof_dir(fold)
    cands = sorted((d for d in (run_dir / "oof").glob(f"fold{fold}_*") if d.is_dir()), key=lambda d: len(list(d.glob("*.npz"))))
    if not cands:
        raise FileNotFoundError(f"No out-of-fold maps for fold {fold} under {run_dir / 'oof'}")
    return cands[-1]

def test_map_of(run_dir, stem):
    hits = sorted((run_dir / "test_probs").glob(f"{stem}_{V3_SHA8}.npz" if run_dir == V3_RUN else f"{stem}.npz"))
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected one test map for {stem} under {run_dir / 'test_probs'}, found {len(hits)}")
    return hits[0]

MAPS_LOCAL = LOCAL_ROOT / "maps" / TAG
_tasks, _owners = [], []
for fold in V3_FOLDS:
    dirs = [oof_dir_of(r, fold) for r in MAP_DIRS]
    names = sorted(FOLDS.loc[FOLDS.fold == fold, "file_name"])
    if DEBUG_LIMIT:
        names = names[:DEBUG_LIMIT]
    for name in names:
        stem = Path(name).stem
        srcs = [d / f"{stem}.npz" for d in dirs]
        for s in srcs:
            if not s.is_file():
                raise FileNotFoundError(f"Missing out-of-fold map {s}")
        _tasks.append((srcs, MAPS_LOCAL / f"oof_fold{fold}" / f"{stem}_fg.npy", MAPS_LOCAL / f"oof_fold{fold}" / f"{stem}_ct.npy"))
        _owners.append(name)
with ThreadPoolExecutor(max_workers=8) as pool:
    OOF_PATHS = dict(zip(_owners, tqdm(pool.map(stage_avg, _tasks), total=len(_tasks), desc="Stage out-of-fold maps")))

_tasks, _owners = [], []
for name in sorted(TEST_PATHS)[:DEBUG_LIMIT] if DEBUG_LIMIT else sorted(TEST_PATHS):
    stem = Path(name).stem
    _tasks.append(([test_map_of(r, stem) for r in MAP_DIRS], MAPS_LOCAL / "test" / f"{stem}_fg.npy", MAPS_LOCAL / "test" / f"{stem}_ct.npy"))
    _owners.append(name)
with ThreadPoolExecutor(max_workers=8) as pool:
    TEST_MAPS = dict(zip(_owners, tqdm(pool.map(stage_avg, _tasks), total=len(_tasks), desc="Stage test maps")))

print("V3 geometry:", GEOM, "| V3 keep threshold:", V3_TAU, "| maps averaged over", len(MAP_DIRS), "run(s)")
print("Maps staged:", len(OOF_PATHS), "out-of-fold,", len(TEST_MAPS), "test")


## Cell 08 — (fusion notebook / V4, unchanged)


In [ ]:
STRUCT8 = ndi.generate_binary_structure(2, 2)
STRUCT4 = ndi.generate_binary_structure(2, 1)
CONN = STRUCT8 if CFG.connectivity == 8 else STRUCT4
FEATURES = ("log_area", "mean_p", "max_p", "frac50", "frac70", "frac90", "mean_contact",
            "elongation", "major_len", "radial", "mean_img", "mean_flat")
F_IDX = {k: i for i, k in enumerate(FEATURES)}

def qt(t):
    return int(round(float(t) * 255))

def match_stats(iou):
    """Strict IoU>0.5, one-to-one max-IoU assignment. Returns (sum_iou, tp, fp, fn, records)."""
    n_gt, n_pred = iou.shape
    if n_gt == 0 or n_pred == 0:
        return np.array([0.0, 0, n_pred, n_gt, 1], np.float64)
    q = iou > 0.5
    if not q.any():
        return np.array([0.0, 0, n_pred, n_gt, 1], np.float64)
    rows, cols = np.flatnonzero(q.any(1)), np.flatnonzero(q.any(0))
    sub = np.where(q[np.ix_(rows, cols)], iou[np.ix_(rows, cols)], 0.0)
    r, c = linear_sum_assignment(sub, maximize=True)
    v = sub[r, c]
    v = v[v > 0]
    tp = len(v)
    return np.array([float(v.sum()), tp, n_pred - tp, n_gt - tp, 1], np.float64)

def match_pairs(iou):
    if iou.size == 0 or not (iou > 0.5).any():
        return []
    sub = np.where(iou > 0.5, iou, 0.0)
    r, c = linear_sum_assignment(sub, maximize=True)
    return [(int(a), int(b), float(sub[a, b])) for a, b in zip(r, c) if sub[a, b] > 0]

def summarize(st):
    s, tp, fp, fn, rec = [float(x) for x in st]
    den = tp + 0.5 * (fp + fn)
    return {"pq": s / den if den else 1.0, "sq": s / tp if tp else 0.0, "rq": tp / den if den else 1.0,
            "precision": tp / (tp + fp) if tp + fp else 0.0, "recall": tp / (tp + fn) if tp + fn else 0.0,
            "tp": int(tp), "fp": int(fp), "fn": int(fn), "records": int(rec)}

def relabel(lab):
    if lab.max() == 0:
        return np.zeros(lab.shape, np.int32), 0
    areas = np.bincount(lab.ravel())
    ids = np.flatnonzero(areas[1:]) + 1
    lut = np.zeros(len(areas), np.int32)
    lut[ids] = np.arange(1, len(ids) + 1, dtype=np.int32)
    return lut[lab], len(ids)

def fill_label_holes(lab):
    fg = lab > 0
    holes = ndi.binary_fill_holes(fg) & ~fg
    if not holes.any():
        return lab
    hl, _ = ndi.label(holes, structure=STRUCT4)
    out = lab.copy()
    for i, sl in enumerate(ndi.find_objects(hl), 1):
        if sl is None:
            continue
        esl = (slice(max(sl[0].start - 1, 0), sl[0].stop + 1), slice(max(sl[1].start - 1, 0), sl[1].stop + 1))
        hole = hl[esl] == i
        ring = ndi.binary_dilation(hole, STRUCT8) & ~hole
        vals = lab[esl][ring]
        vals = vals[vals > 0]
        if vals.size:
            out[esl][hole] = np.bincount(vals).argmax()
    return out

def geometry_labels(fg, ct, geom):
    t_low, contact_t, offset = geom
    low = fg >= qt(t_low)
    if contact_t is not None:
        cores = low & (ct < qt(contact_t))
        lab, n = ndi.label(cores, structure=CONN)
        if n:
            areas = np.bincount(lab.ravel(), minlength=n + 1)
            small = areas < CFG.min_core_area
            small[0] = False
            if small.any():
                lab[small[lab]] = 0
        if lab.any():
            dist, (iy, ix) = ndi.distance_transform_edt(lab == 0, return_distances=True, return_indices=True)
            lab = np.where(low & (dist <= CFG.contact_assign_radius), lab[iy, ix], 0).astype(np.int32)
        else:
            lab = np.zeros((fg.shape), np.int32)
    else:
        lab, _ = ndi.label(low, structure=CONN)
        lab = lab.astype(np.int32)
    if CFG.fill_holes and lab.any():
        lab = fill_label_holes(lab)
    for _ in range(max(offset, 0)):
        lab = np.where(lab == 0, ndi.grey_dilation(lab, footprint=STRUCT4), lab)
    for _ in range(max(-offset, 0)):
        lab = np.where(ndi.grey_erosion(lab, footprint=STRUCT4) == lab, lab, 0)
    return relabel(lab)

def component_features(lab, n, fg, ct, img, flat):
    if n == 0:
        return np.zeros((0, len(FEATURES)), np.float32), np.zeros(0, np.float64)
    L = lab.ravel()
    nz = np.flatnonzero(L)
    Lz = L[nz]
    def s(w):
        return np.bincount(Lz, weights=w, minlength=n + 1)[1:]
    area = np.bincount(Lz, minlength=n + 1)[1:].astype(np.float64)
    a = np.maximum(area, 1.0)
    fgz = np.asarray(fg).ravel()[nz]
    p = fgz.astype(np.float64) / 255.0
    mx = np.zeros(n + 1, np.float64)
    np.maximum.at(mx, Lz, fgz.astype(np.float64))
    yz, xz = YY_FLAT[nz], XX_FLAT[nz]
    my, mxx = s(yz) / a, s(xz) / a
    syy = np.maximum(s(yz * yz) / a - my ** 2, 0)
    sxx = np.maximum(s(xz * xz) / a - mxx ** 2, 0)
    sxy = s(yz * xz) / a - my * mxx
    tr, det = syy + sxx, syy * sxx - sxy ** 2
    disc = np.sqrt(np.maximum(tr ** 2 / 4 - det, 0))
    l1, l2 = tr / 2 + disc, np.maximum(tr / 2 - disc, 1e-3)
    feats = np.stack([
        np.log1p(area), s(p) / a, mx[1:] / 255.0,
        s((fgz >= qt(0.5)).astype(np.float64)) / a, s((fgz >= qt(0.7)).astype(np.float64)) / a,
        s((fgz >= qt(0.9)).astype(np.float64)) / a,
        s(np.asarray(ct).ravel()[nz].astype(np.float64) / 255.0) / a,
        np.sqrt((l1 + 1e-3) / l2), 4.0 * np.sqrt(l1),
        np.hypot(my - CY, mxx - CX) / DISK_R,
        s(np.asarray(img).ravel()[nz].astype(np.float64) / 255.0) / a,
        s(np.asarray(flat).ravel()[nz].astype(np.float64) / 128.0) / a,
    ], 1).astype(np.float32)
    return feats, area

def overlap_matrices(lab, n, area, gt):
    L = lab.ravel()
    out = []
    for rec in gt["records"]:
        inter = np.zeros((len(rec["idx"]), n), np.float64)
        if n:
            for j, idx in enumerate(rec["idx"]):
                inter[j] = np.bincount(L[idx], minlength=n + 1)[1:]
        union = rec["area"][:, None] + area[None, :] - inter
        iou = np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)
        out.append((inter, iou))
    return out

def grid_keep(feats, area, flt):
    th, mc, ma = flt
    if len(area) == 0:
        return np.zeros(0, bool)
    return ((np.rint(feats[:, F_IDX["max_p"]] * 255) >= qt(th)) & (feats[:, F_IDX["mean_p"]] >= mc - 1e-9)
            & (area >= ma))

def prefilter_keep(feats, area):
    if len(area) == 0:
        return np.zeros(0, bool)
    return ((np.rint(feats[:, F_IDX["max_p"]] * 255) >= qt(CFG.scorer_prefilter_max))
            & (area >= CFG.scorer_prefilter_area))

def load_maps(fg, ct):
    if isinstance(fg, (str, Path)):
        fg = np.load(fg, mmap_mode="r")
    if isinstance(ct, (str, Path)):
        ct = np.load(ct, mmap_mode="r")
    return np.asarray(fg), np.asarray(ct)

# ---- process-pool workers (fork: they see GT_INDEX, CACHE, CFG of the parent at fork time) ----
def score_image_grid(task):
    name, fg, ct, geoms, filters = task
    fg, ct = load_maps(fg, ct)
    img, flat = cached_arrays(cache_key("train", name), False)
    gt = GT_INDEX[name]
    out = np.zeros((len(geoms), len(filters), 5), np.float64)
    for gi, g in enumerate(geoms):
        lab, n = geometry_labels(fg, ct, g)
        feats, area = component_features(lab, n, fg, ct, img, flat)
        mats = overlap_matrices(lab, n, area, gt)
        for fi, flt in enumerate(filters):
            keep = grid_keep(feats, area, flt)
            for _, iou in mats:
                out[gi, fi] += match_stats(iou[:, keep])
    return name, out

def scorer_rows(task):
    name, fg, ct, geom = task
    fg, ct = load_maps(fg, ct)
    img, flat = cached_arrays(cache_key("train", name), False)
    lab, n = geometry_labels(fg, ct, geom)
    feats, area = component_features(lab, n, fg, ct, img, flat)
    mats = overlap_matrices(lab, n, area, GT_INDEX[name])
    y = np.zeros(n, np.float64)
    for _, iou in mats:
        if iou.shape[0] and n:
            b = iou.max(0)
            y += np.where(b > 0.5, b, 0.0)
    y /= max(len(mats), 1)
    pre = prefilter_keep(feats, area)
    return name, feats[pre], area[pre], y[pre], [iou[:, pre] for _, iou in mats]

_POOL = None
def get_pool(refresh=False):
    """Fork-based CPU pool. Refresh after defining new globals that workers must see."""
    global _POOL
    if refresh and _POOL is not None:
        _POOL.shutdown(wait=True)
        _POOL = None
    if _POOL is None:
        _POOL = ProcessPoolExecutor(max_workers=CPU_POOL, mp_context=mp.get_context("fork"))
    return _POOL

def stats_table(total, geoms, filters):
    rows = []
    for gi, g in enumerate(geoms):
        for fi, f in enumerate(filters):
            rows.append({"t_low": g[0], "contact": g[1], "offset": g[2], "t_high": f[0], "min_mean": f[1],
                         "min_area": f[2], **summarize(total[gi, fi])})
    return pd.DataFrame(rows)

def pick_best(table):
    t = table.copy()
    t["_c"] = t["contact"].isna()
    t["_o"] = (t["offset"] == 0)
    t = t.sort_values(["pq", "_c", "_o", "min_area", "min_mean"], ascending=[False, False, False, True, True])
    return t.iloc[0].drop(["_c", "_o"]).to_dict()

def as_geom(row):
    c = row["contact"]
    return (float(row["t_low"]), None if (c is None or (isinstance(c, float) and np.isnan(c))) else float(c), int(row["offset"]))

def as_filter(row):
    return (float(row["t_high"]), float(row["min_mean"]), int(row["min_area"]))

print("PQ engine ready. Features:", FEATURES)


## Cell 09 — (fusion notebook / V4, unchanged)


In [ ]:
DET_FOLDS, DET_OOF, _test_by_fold = [], {}, {}
for fold in range(CFG.n_folds):
    oof_f, test_f = sorted(DET_DIR.glob(f"oof_fold{fold}_*.pkl")), sorted(DET_DIR.glob(f"test_fold{fold}_*.pkl"))
    if len(oof_f) != 1 or len(test_f) != 1:
        raise FileNotFoundError(f"Expected one oof and one test detection file for fold {fold} in {DET_DIR}")
    with open(oof_f[0], "rb") as f:
        DET_OOF.update(pickle.load(f))
    with open(test_f[0], "rb") as f:
        _test_by_fold[fold] = pickle.load(f)
    DET_FOLDS.append(fold)
DET_TEST = {n: [_test_by_fold[f][n] for f in DET_FOLDS] for n in sorted(TEST_MAPS)}
print("Detectors:", DET_FOLDS, "| out-of-fold images with detections:", len(DET_OOF),
      "| mean detections per image:", round(float(np.mean([len(d['scores']) for d in DET_OOF.values()])), 1))


## Cell 10 — V4 fusion: detector evidence, detector-only candidates, build_candidates (unchanged)


In [ ]:
DET_FEATURES = ("det_score", "det_iou", "det_cov")
ALL_FEATURES = tuple(FEATURES) + DET_FEATURES + ("t_low", "src")     # V5: t_low = hysteresis level of the candidate
N_V3F, N_DETF = len(FEATURES), len(DET_FEATURES)
_GROW_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * CFG.det_grow + 1, 2 * CFG.det_grow + 1))

def rasterize_detections(dets):
    """Detections of one detector -> list of (score, y0, x0, boolean mask of its bounding window), best first."""
    out = []
    for s, p in zip(dets["scores"], dets["polys"]):
        if p is None or len(p) < 3:
            continue
        q = np.rint(np.asarray(p, np.float64)).astype(np.int32)
        x0, y0 = max(int(q[:, 0].min()), 0), max(int(q[:, 1].min()), 0)
        x1, y1 = min(int(q[:, 0].max()) + 1, W), min(int(q[:, 1].max()) + 1, H)
        if x1 - x0 < 2 or y1 - y0 < 2:
            continue
        roi = np.zeros((y1 - y0, x1 - x0), np.uint8)
        cv2.fillPoly(roi, [q - np.array([x0, y0], np.int32)], 1)
        if int(roi.sum()) < CFG.det_min_area:
            continue
        out.append((float(s), y0, x0, roi.astype(bool)))
    out.sort(key=lambda t: -t[0])
    return out

def detection_evidence(lab, n, area, rast):
    """Per candidate: score of the detection sharing most pixels with it, IoU with that detection, fraction covered."""
    score, iou, cover = np.zeros(n), np.zeros(n), np.zeros(n)
    if n == 0 or not rast:
        return score, iou, cover
    best = np.zeros(n)
    covered = np.zeros(lab.shape, bool)
    for s, y0, x0, m in rast:
        h, w = m.shape
        covered[y0:y0 + h, x0:x0 + w] |= m
        inter = np.bincount(lab[y0:y0 + h, x0:x0 + w][m], minlength=n + 1)[1:].astype(np.float64)
        a = float(m.sum())
        upd = (inter >= CFG.det_link_frac * np.minimum(area, a)) & (inter > best)
        if upd.any():
            best[upd] = inter[upd]
            score[upd] = s
            iou[upd] = inter[upd] / (area[upd] + a - inter[upd])
    cover = np.bincount(lab[covered], minlength=n + 1)[1:] / np.maximum(area, 1.0)
    return score, iou, cover

def largest_piece(mask):
    lab, n = ndi.label(mask, structure=CONN)
    if n <= 1:
        return mask
    sizes = np.bincount(lab.ravel())
    sizes[0] = 0
    return lab == int(sizes.argmax())

def consensus_scores(rasts):
    """Every detection gets the mean, over all detectors, of the best score among that detector's detections
    that overlap it (IoU >= 0.3). With one detector this is its own score; with four, a detection found by only
    one of them is worth a quarter. Keeps test images (4 detectors) comparable to out-of-fold images (1 detector)."""
    items = [(d, s, y0, x0, m, float(m.sum())) for d, rast in enumerate(rasts) for s, y0, x0, m in rast]
    out = []
    for d, s, y0, x0, m, a in items:
        total = s
        for d2 in range(len(rasts)):
            if d2 == d:
                continue
            best = 0.0
            for s2, v0, u0, m2 in rasts[d2]:
                if s2 <= best:
                    break                                         # sorted by score: nothing better follows
                ya, yb = max(y0, v0), min(y0 + m.shape[0], v0 + m2.shape[0])
                xa, xb = max(x0, u0), min(x0 + m.shape[1], u0 + m2.shape[1])
                if ya >= yb or xa >= xb:
                    continue
                inter = float((m[ya - y0:yb - y0, xa - x0:xb - x0] & m2[ya - v0:yb - v0, xa - u0:xb - u0]).sum())
                if inter >= 0.3 * (a + float(m2.sum()) - inter):
                    best = s2
            total += best
        out.append((total / max(len(rasts), 1), y0, x0, m))
    out.sort(key=lambda t: -t[0])
    return out

def detector_only_candidates(lab, fg, rasts):
    """Objects the detectors found where the UNet has no candidate. The mask is the UNet's probability inside the
    (slightly grown) detection when there is enough of it, otherwise the detection itself. All disjoint."""
    pooled = consensus_scores(rasts)
    occupied = lab > 0
    new = np.zeros(lab.shape, np.int32)
    kinds, g = [], CFG.det_grow
    for s, y0, x0, m in pooled:
        if s < CFG.det_only_floor:
            break
        h, w = m.shape
        if int((m & occupied[y0:y0 + h, x0:x0 + w]).sum()) > CFG.det_only_max_claimed * int(m.sum()):
            continue                                              # already represented
        ya, xa, yb, xb = max(y0 - g, 0), max(x0 - g, 0), min(y0 + h + g, H), min(x0 + w + g, W)
        big = np.zeros((yb - ya, xb - xa), np.uint8)
        big[y0 - ya:y0 - ya + h, x0 - xa:x0 - xa + w] = m
        free = ~occupied[ya:yb, xa:xb]
        det_free = (big > 0) & free
        claim = (cv2.dilate(big, _GROW_KERNEL) > 0) & free & (fg[ya:yb, xa:xb] >= qt(CFG.det_claim_t))
        if int(claim.sum()) >= CFG.det_claim_min_frac * int(det_free.sum()):
            mask, kind = largest_piece(claim), 1                  # drawn by the UNet
        else:
            mask, kind = largest_piece(det_free), 2               # drawn by the detector
        if int(mask.sum()) < CFG.det_only_min_area:
            continue
        kinds.append(kind)
        new[ya:yb, xa:xb][mask] = len(kinds)
        occupied[ya:yb, xa:xb] |= mask
    return new, np.asarray(kinds, np.float32)

def build_candidates(fg, ct, img, flat, det_list, with_detector_only=True):
    """V3 candidates (same geometry and prefilter as V3) + detector-only candidates, with all features.
    Returns (label map, n candidates, n from V3, feature matrix [n, len(ALL_FEATURES)], areas)."""
    lab0, n0 = geometry_labels(fg, ct, GEOM)
    feats, area = component_features(lab0, n0, fg, ct, img, flat)
    pre = prefilter_keep(feats, area)
    lut = np.zeros(n0 + 1, np.int32)
    ids = np.flatnonzero(pre) + 1
    lut[ids] = np.arange(1, len(ids) + 1, dtype=np.int32)
    lab, n_v3 = lut[lab0], len(ids)
    feats, area = feats[pre], area[pre]
    rasts = [rasterize_detections(d) for d in det_list]
    kinds = np.zeros(0, np.float32)
    if with_detector_only and any(rasts):
        new, kinds = detector_only_candidates(lab, fg, rasts)
        if len(kinds):
            lab = np.where(new > 0, new + n_v3, lab)
            feats, area = component_features(lab, n_v3 + len(kinds), fg, ct, img, flat)
    n = n_v3 + len(kinds)
    ev = np.zeros((n, N_DETF))
    for rast in rasts:
        ev += np.stack(detection_evidence(lab, n, area, rast), 1)
    if rasts:
        ev /= len(rasts)
    src = np.r_[np.zeros(n_v3, np.float32), kinds]
    lev = np.full((n, 1), GEOM[0], np.float32)
    X = np.concatenate([feats.astype(np.float32), ev.astype(np.float32), lev, src[:, None].astype(np.float32)], 1)
    return lab, n, n_v3, X, area

def level_rows(fg, ct, img, flat, det_list, gt):
    """V5 step 1: candidates at the extra hysteresis levels (same contact / offset / prefilter, no detector-only),
    with their features and labels. Used only as additional training rows for the keep-scorer."""
    rasts = [rasterize_detections(d) for d in det_list]
    Xs, ys = [], []
    for t in CFG.aug_levels:
        lab0, n0 = geometry_labels(fg, ct, (float(t), GEOM[1], GEOM[2]))
        feats, area = component_features(lab0, n0, fg, ct, img, flat)
        pre = prefilter_keep(feats, area)
        if not pre.any():
            continue
        lut = np.zeros(n0 + 1, np.int32)
        ids = np.flatnonzero(pre) + 1
        lut[ids] = np.arange(1, len(ids) + 1, dtype=np.int32)
        lab, n = lut[lab0], len(ids)
        feats, area = feats[pre], area[pre]
        ev = np.zeros((n, N_DETF))
        for rast in rasts:
            ev += np.stack(detection_evidence(lab, n, area, rast), 1)
        if rasts:
            ev /= len(rasts)
        mats = overlap_matrices(lab, n, area, gt)
        y = np.zeros(n)
        for _, iou in mats:
            if iou.shape[0]:
                b = iou.max(0)
                y += np.where(b > 0.5, b, 0.0)
        y /= max(len(mats), 1)
        Xs.append(np.concatenate([feats.astype(np.float32), ev.astype(np.float32), np.full((n, 1), t, np.float32),
                                  np.zeros((n, 1), np.float32)], 1))
        ys.append(y)
    if not Xs:
        return np.zeros((0, len(ALL_FEATURES)), np.float32), np.zeros(0)
    return np.concatenate(Xs), np.concatenate(ys)

def fusion_tests():
    global GEOM
    saved, GEOM = GEOM, (0.4, None, 0)                             # fixed geometry for the test only
    try:
        _fusion_tests()
    finally:
        GEOM = saved

def _fusion_tests():
    fg = np.zeros((H, W), np.uint8); ct = np.zeros((H, W), np.uint8)
    img = np.full((H, W), 120, np.uint8); flat = np.full((H, W), 128, np.uint8)
    fg[100:110, 100:300] = 230                                     # A: UNet candidate, also detected
    fg[200:210, 100:300] = 220                                     # B: UNet candidate, not detected
    fg[300:308, 100:260] = 70                                      # C: below the UNet threshold, detected -> UNet-drawn
    box = lambda y0, y1, x0, x1: np.array([[x0, y0], [x1, y0], [x1, y1], [x0, y1]], np.float32)
    dets = {"scores": np.array([0.9, 0.6, 0.4, 0.03], np.float32),
            "polys": [box(99, 110, 98, 301), box(299, 308, 98, 262), box(400, 409, 100, 250), box(450, 459, 100, 250)]}
    lab, n, n_v3, X, area = build_candidates(fg, ct, img, flat, [dets])
    assert n_v3 == 2 and n == 4, (n_v3, n)                         # A, B + C (UNet-drawn) + D (detector-drawn); 0.03 ignored
    col = {k: i for i, k in enumerate(ALL_FEATURES)}
    a, b = lab[105, 200] - 1, lab[205, 200] - 1
    assert X[a, col["det_score"]] > 0.85 and X[a, col["det_iou"]] > 0.7 and X[a, col["det_cov"]] > 0.95
    assert X[b, col["det_score"]] == 0 and X[b, col["det_cov"]] == 0
    c, d = lab[304, 150] - 1, lab[404, 150] - 1
    assert X[c, col["src"]] == 1 and X[d, col["src"]] == 2 and lab[454, 150] == 0
    assert abs(X[c, col["det_score"]] - 0.6) < 1e-6 and abs(X[d, col["det_score"]] - 0.4) < 1e-6
    lab2, n2, _, X2, _ = build_candidates(fg, ct, img, flat, [dets, {"scores": np.zeros(0, np.float32), "polys": []}])
    assert n2 == 4 and abs(X2[a, col["det_score"]] - X[a, col["det_score"]] / 2) < 1e-6     # mean over detectors
    lab3, n3, n3v, X3, _ = build_candidates(fg, ct, img, flat, [dets], with_detector_only=False)
    assert n3 == n3v == 2 and np.allclose(X3[:, :N_V3F + N_DETF], X[:2, :N_V3F + N_DETF])
    weak = {"scores": np.array([0.15], np.float32), "polys": [box(400, 409, 100, 250)]}
    none = {"scores": np.zeros(0, np.float32), "polys": []}
    assert build_candidates(fg, ct, img, flat, [weak])[1] == 3                 # one detector at 0.15: candidate
    assert build_candidates(fg, ct, img, flat, [weak, none])[1] == 2           # found by 1 of 2: 0.075 < floor, none
    assert build_candidates(fg, ct, img, flat, [weak, weak])[1] == 3           # found by both: candidate again
    for k in range(1, n + 1):                                      # every candidate is one non-empty, disjoint region
        assert (lab == k).any()
    print("PASS: detector evidence, detector-only candidates (UNet-drawn and detector-drawn), consensus over detectors")

fusion_tests()


## Cell 11 — V4 candidates of every image, saved label maps, and each candidate's annotated counterparts


In [ ]:
PILOT = sorted(n for n in OOF_PATHS if FOLD_OF[n] == CFG.pilot_fold)
TRAIN = sorted(n for n in OOF_PATHS if FOLD_OF[n] != CFG.pilot_fold)
if DEBUG:
    PILOT, TRAIN = PILOT[:12], TRAIN[:40]
LAB_DIR = LOCAL_ROOT / "labels_v4"
LAB_DIR.mkdir(parents=True, exist_ok=True)

def cand_task(name):
    """V4 candidates (exactly as the fusion run builds them) -> label map on disk, windows, counterparts per candidate."""
    fg, ct = load_maps(*OOF_PATHS[name])
    img, flat = cached_arrays(cache_key("train", name), False)
    lab, n, n_v3, X, area = build_candidates(fg, ct, img, flat, [DET_OOF[name]])
    save_npy_atomic(LAB_DIR / f"{Path(name).stem}.npy", lab.astype(np.uint16))
    wins = []
    for k, sl in enumerate(ndi.find_objects(lab, max_label=n)):
        wins.append((sl[0].start, sl[1].start, lab[sl] == k + 1) if sl is not None else None)
    mats = overlap_matrices(lab, n, area, GT_INDEX[name])
    members = [[] for _ in range(n)]
    for r, (_, iou) in enumerate(mats):
        if iou.shape[0] == 0:
            continue
        j = iou.argmax(0)
        for c in range(n):
            if iou[j[c], c] >= CFG.match_min_iou:
                members[c].append((r, int(j[c]), float(iou[j[c], c])))
    return name, n, wins, members

pool = get_pool(refresh=True)
CANDS = {}
for name, n, wins, members in tqdm(pool.map(cand_task, TRAIN + PILOT, chunksize=2), total=len(TRAIN) + len(PILOT),
                                   desc="V4 candidates"):
    CANDS[name] = (n, wins, members)

def gt_groups(name):
    """For each (record, instance): the matching instance in every annotator set (itself included)."""
    recs = GT_INDEX[name]["records"]
    labels = []
    for rec in recs:
        v = np.zeros(H * W, np.int32)
        for j, idx in enumerate(rec["idx"]):
            v[idx] = j + 1
        labels.append(v)
    groups = {}
    for r, rec in enumerate(recs):
        for j, idx in enumerate(rec["idx"]):
            g = [(r, j, 1.0)]
            for r2, other in enumerate(recs):
                if r2 == r or not len(other["idx"]):
                    continue
                cnt = np.bincount(labels[r2][idx], minlength=len(other["idx"]) + 1)[1:].astype(np.float64)
                iou = cnt / (rec["area"][j] + other["area"] - cnt)
                if iou.max() >= CFG.match_min_iou:
                    g.append((r2, int(iou.argmax()), float(iou.max())))
            groups[(r, j)] = g
    return name, groups

pool = get_pool(refresh=True)                     # workers must see gt_groups
GROUPS = dict(tqdm(pool.map(gt_groups, TRAIN, chunksize=4), total=len(TRAIN), desc="Annotated counterparts"))
TRAIN_REAL = [(n, k) for n in TRAIN for k in range(CANDS[n][0]) if CANDS[n][2][k] and CANDS[n][1][k] is not None]
TRAIN_GT = [(n, r, j) for n in TRAIN for (r, j) in GROUPS[n]]
print("Pilot images:", len(PILOT), "| training images:", len(TRAIN), "| V4 candidates per image:",
      round(np.mean([CANDS[n][0] for n in PILOT + TRAIN]), 2))
print("Training seeds: real candidates with an annotated counterpart", len(TRAIN_REAL), "| annotated filaments", len(TRAIN_GT))


## Cell 12 — Rotated frames: candidate -> 384 x 128 crop along its long axis, and back


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

CW, CH = CFG.crop_w, CFG.crop_h

def frame_of(ys, xs, rng=None):
    """Pixel coordinates of a candidate -> (centre y, centre x, angle, native length). Length:width = CW:CH."""
    cy, cx = ys.mean(), xs.mean()
    if len(ys) > 2:
        cov = np.cov(np.stack([xs - cx, ys - cy]))
        ev, evec = np.linalg.eigh(cov)
        vx, vy = evec[:, 1]
    else:
        vx, vy = 1.0, 0.0
    th = math.atan2(vy, vx)
    a = (xs - cx) * math.cos(th) + (ys - cy) * math.sin(th)
    b = -(xs - cx) * math.sin(th) + (ys - cy) * math.cos(th)
    mid_a, mid_b = (a.max() + a.min()) / 2, (b.max() + b.min()) / 2
    cx, cy = cx + mid_a * math.cos(th) - mid_b * math.sin(th), cy + mid_a * math.sin(th) + mid_b * math.cos(th)
    length = max(CFG.pad_along * (a.max() - a.min() + 1), CFG.min_len)
    length = max(length, (b.max() - b.min() + 1 + 2 * CFG.across_margin) * CW / CH)
    if rng is not None:
        th += math.radians(rng.uniform(-10, 10))
        length *= rng.uniform(0.9, 1.2)
        cx += rng.uniform(-0.05, 0.05) * length; cy += rng.uniform(-0.05, 0.05) * length * CH / CW
    return cy, cx, th, length

def frame_matrix(cy, cx, th, length):
    """2x3 affine: crop pixel (u, v) -> image pixel (x, y)."""
    s = length / CW
    c, n = math.cos(th) * s, math.sin(th) * s
    return np.array([[c, -n, cx - c * (CW - 1) / 2 + n * (CH - 1) / 2],
                     [n, c, cy - n * (CW - 1) / 2 - c * (CH - 1) / 2]], np.float64)

def frame_box(M, pad=2):
    pts = M @ np.array([[0, CW - 1, 0, CW - 1], [0, 0, CH - 1, CH - 1], [1, 1, 1, 1]], np.float64)
    x0, y0 = int(math.floor(pts[0].min())) - pad, int(math.floor(pts[1].min())) - pad
    x1, y1 = int(math.ceil(pts[0].max())) + pad + 1, int(math.ceil(pts[1].max())) + pad + 1
    return y0, x0, y1, x1

def window(arr, box, fill):
    y0, x0, y1, x1 = box
    out = np.full((y1 - y0, x1 - x0), fill, np.float32)
    ya, xa, yb, xb = max(y0, 0), max(x0, 0), min(y1, H), min(x1, W)
    if ya < yb and xa < xb:
        out[ya - y0:yb - y0, xa - x0:xb - x0] = arr[ya:yb, xa:xb]
    return out

def to_crop(win, M, box, interp=cv2.INTER_LINEAR, fill=0.0):
    Mw = M.copy(); Mw[0, 2] -= box[1]; Mw[1, 2] -= box[0]
    return cv2.warpAffine(win, Mw, (CW, CH), flags=interp | cv2.WARP_INVERSE_MAP, borderMode=cv2.BORDER_CONSTANT,
                          borderValue=fill)

def to_window(crop, M, box):
    """Crop-frame map -> the frame's bounding window in image coordinates."""
    Mw = M.copy(); Mw[0, 2] -= box[1]; Mw[1, 2] -= box[0]
    inv = cv2.invertAffineTransform(Mw)
    return cv2.warpAffine(crop.astype(np.float32), inv, (box[3] - box[1], box[2] - box[0]),
                          flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP, borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)

def idx_window(idx, box):
    y0, x0, y1, x1 = box
    out = np.zeros((y1 - y0, x1 - x0), np.float32)
    ys, xs = idx // W - y0, idx % W - x0
    ok = (ys >= 0) & (ys < y1 - y0) & (xs >= 0) & (xs < x1 - x0)
    out[ys[ok], xs[ok]] = 1.0
    return out

def seed_window(seed, box):
    y0, x0, m = seed
    return idx_window((np.flatnonzero(m) // m.shape[1] + y0) * W + (np.flatnonzero(m) % m.shape[1] + x0), box)

def make_input(name, seed, others_lab_ids, M, box):
    """5 channels: raw, flattened, V3 probability, this candidate, every other candidate."""
    img, flat = cached_arrays(cache_key("train", name), False)
    fg = np.load(OOF_PATHS[name][0], mmap_mode="r")
    lab = np.load(LAB_DIR / f"{Path(name).stem}.npy", mmap_mode="r")
    lw = window(lab, box, 0)
    sw = seed_window(seed, box)
    other = ((lw > 0) & ~np.isin(lw, others_lab_ids) & (sw < 0.5)).astype(np.float32)
    x = np.stack([(to_crop(window(img, box, 0), M, box) / 255.0 - 0.5) / 0.25,
                  (to_crop(window(flat, box, 128), M, box, fill=128.0) / 128.0 - 1.0) / 0.15,
                  to_crop(window(fg, box, 0), M, box) / 255.0,
                  to_crop(sw, M, box),
                  to_crop(other, M, box)])
    return x.astype(np.float32)

def make_target(name, members, M, box):
    """P(marked | drawn): mean over the annotator sets that drew this filament."""
    recs = GT_INDEX[name]["records"]
    t = np.zeros((box[2] - box[0], box[3] - box[1]), np.float32)
    for r, j, _ in members:
        t += idx_window(recs[r]["idx"][j], box)
    t /= max(len(members), 1)
    return to_crop(t, M, box)[None]

def seed_pixels(seed):
    y0, x0, m = seed
    ys, xs = np.nonzero(m)
    return ys + y0, xs + x0

def jitter_gt_seed(name, r, j, rng):
    idx = GT_INDEX[name]["records"][r]["idx"][j]
    ys, xs = idx // W, idx % W
    y0, x0 = ys.min(), xs.min()
    m = np.zeros((ys.max() - y0 + 1, xs.max() - x0 + 1), np.uint8)
    m[ys - y0, xs - x0] = 1
    k = int(rng.integers(-2, 3))
    if k:
        pad = abs(k) + 1
        m = np.pad(m, pad); y0 -= pad; x0 -= pad
        ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * abs(k) + 1, 2 * abs(k) + 1))
        m = cv2.dilate(m, ker) if k > 0 else cv2.erode(m, ker)
    if not m.any():
        m[m.shape[0] // 2, m.shape[1] // 2] = 1
    dy, dx = rng.integers(-3, 4, size=2)
    return int(y0 + dy), int(x0 + dx), m.astype(bool)

class CropTrain(Dataset):
    def __init__(self, n, epoch):
        self.n, self.epoch = n, epoch
    def __len__(self):
        return self.n
    def __getitem__(self, i):
        rng = np.random.default_rng((CFG.seed, self.epoch, i))
        if rng.random() < CFG.gt_seed_fraction:
            name, r, j = TRAIN_GT[int(rng.integers(len(TRAIN_GT)))]
            seed, members = jitter_gt_seed(name, r, j, rng), GROUPS[name][(r, j)]
            lab = np.load(LAB_DIR / f"{Path(name).stem}.npy", mmap_mode="r")
            hit = np.unique(np.asarray(lab).ravel()[GT_INDEX[name]["records"][r]["idx"][j]])
            own = hit[hit > 0]                                    # candidates on this filament are not "others"
        else:
            name, k = TRAIN_REAL[int(rng.integers(len(TRAIN_REAL)))]
            seed, members, own = CANDS[name][1][k], CANDS[name][2][k], np.array([k + 1])
        cy, cx, th, length = frame_of(*seed_pixels(seed), rng=rng)
        M = frame_matrix(cy, cx, th, length); box = frame_box(M)
        x, y = make_input(name, seed, own, M, box), make_target(name, members, M, box)
        if rng.random() < 0.5:
            x, y = x[:, :, ::-1], y[:, :, ::-1]
        if rng.random() < 0.5:
            x, y = x[:, ::-1, :], y[:, ::-1, :]
        if rng.random() < 0.5:
            x[:2] = x[:2] * rng.uniform(0.85, 1.15) + rng.uniform(-0.15, 0.15)
        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(np.ascontiguousarray(y))

def _geometry_test(n=200):
    """An exact annotated filament as seed: crop -> back must reproduce it; its target must coincide with the seed."""
    rng = np.random.default_rng(1)
    rt, tg = [], []
    for _ in range(n):
        name, r, j = TRAIN_GT[int(rng.integers(len(TRAIN_GT)))]
        idx = GT_INDEX[name]["records"][r]["idx"][j]
        ys, xs = idx // W, idx % W
        m = np.zeros((ys.max() - ys.min() + 1, xs.max() - xs.min() + 1), bool); m[ys - ys.min(), xs - xs.min()] = True
        seed = (int(ys.min()), int(xs.min()), m)
        M = frame_matrix(*frame_of(ys, xs)); box = frame_box(M)
        x, y = make_input(name, seed, np.array([0]), M, box), make_target(name, [(r, j, 1.0)], M, box)
        back = to_window(x[3], M, box) >= 0.5
        ref = seed_window(seed, box) > 0.5
        rt.append((back & ref).sum() / max((back | ref).sum(), 1))
        tg.append(((x[3] >= 0.5) & (y[0] >= 0.5)).sum() / max(((x[3] >= 0.5) | (y[0] >= 0.5)).sum(), 1))
    print(f"Geometry test ({n} filaments): IoU(seed after crop and back, seed) {np.mean(rt):.3f} (min {np.min(rt):.3f});"
          f" IoU(seed, target) in the crop {np.mean(tg):.3f}")
    assert np.mean(rt) > 0.85 and np.mean(tg) > 0.9, "rotated-frame geometry is wrong"

_geometry_test()
_probe = CropTrain(2, 0)[0]
print("Crop tensors:", tuple(_probe[0].shape), tuple(_probe[1].shape))


## Cell 13 — Refiner: UNet / EfficientNetV2-S, 5 channels, 0.5 BCE + Lovász hinge (bf16, channels-last)


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("Training the refiner needs a GPU runtime (L4).")
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
FOLD_DIR = RUN / f"fold{CFG.pilot_fold}"
FOLD_DIR.mkdir(parents=True, exist_ok=True)
CFG.workers = min(CFG.workers, max(2, CPU_COUNT - 1))

def build_model(pretrained=True):
    return smp.Unet(encoder_name=CFG.encoder, encoder_weights="imagenet" if pretrained else None,
                    in_channels=5, classes=1, decoder_attention_type="scse")

def lovasz_grad(gt_sorted):
    gts = gt_sorted.sum()
    inter = gts - gt_sorted.cumsum(0)
    union = gts + (1.0 - gt_sorted).cumsum(0)
    j = 1.0 - inter / union
    if gt_sorted.numel() > 1:
        j[1:] = j[1:] - j[:-1]
    return j

def lovasz_hinge(logits, labels):
    """Per crop (each crop is one filament), averaged."""
    losses = []
    for lg, lb in zip(logits.reshape(len(logits), -1), labels.reshape(len(labels), -1)):
        signs = 2.0 * lb - 1.0
        err = 1.0 - lg * signs
        es, perm = torch.sort(err, descending=True)
        losses.append(torch.dot(F.relu(es), lovasz_grad(lb[perm])))
    return torch.stack(losses).mean()

def loss_fn(out, y):
    out = out.float()
    return 0.5 * F.binary_cross_entropy_with_logits(out, y) + lovasz_hinge(out, (y >= 0.5).float())

def train_refiner():
    done = FOLD_DIR / "done.json"
    if done.is_file() and (FOLD_DIR / "best.pt").is_file():
        print("Refiner already trained:", json.loads(done.read_text()))
        return
    model = build_model().to(DEVICE).to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    steps = CFG.epochs * (CFG.samples_per_epoch // CFG.batch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG.lr, total_steps=steps, pct_start=0.1)
    hist, t0 = [], time.time()
    for ep in range(CFG.epochs):
        model.train()
        dl = DataLoader(CropTrain(CFG.samples_per_epoch, ep), batch_size=CFG.batch, shuffle=False,
                        num_workers=CFG.workers, pin_memory=True, drop_last=True, persistent_workers=False)
        tot, nb, tw = 0.0, 0, time.time()
        for x, y in dl:
            x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last); y = y.to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE, dtype=AMP):
                out = model(x)
            loss = loss_fn(out, y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            tot += float(loss); nb += 1
        hist.append({"epoch": ep + 1, "loss": tot / max(nb, 1), "seconds": round(time.time() - tw, 1)})
        print(f"epoch {ep + 1}/{CFG.epochs} loss {tot / max(nb, 1):.4f} ({time.time() - tw:.0f}s)", flush=True)
        pd.DataFrame(hist).to_csv(FOLD_DIR / "history.csv", index=False)
    torch.save({"state": model.state_dict(), "config": asdict(CFG)}, FOLD_DIR / "best.pt")
    write_json(done, {"fold": CFG.pilot_fold, "minutes": (time.time() - t0) / 60, "epochs": CFG.epochs,
                      "sha256": hash_file(FOLD_DIR / "best.pt")})
    push(f"fold{CFG.pilot_fold}")
    del model, opt
    gc.collect(); torch.cuda.empty_cache()

train_refiner()


## Cell 14 — Refine every pilot-fold candidate (flip TTA), threshold t* = J*/(1+J*)


In [ ]:
MODEL = build_model(pretrained=False)
MODEL.load_state_dict(torch.load(FOLD_DIR / "best.pt", map_location="cpu")["state"])
MODEL = MODEL.to(DEVICE).to(memory_format=torch.channels_last).eval()

@torch.no_grad()
def predict(xs):
    acc = None
    for fh in (False, True):
        for fv in (False, True):
            xb = torch.from_numpy(np.ascontiguousarray(xs)).to(DEVICE)
            if fh: xb = torch.flip(xb, (-1,))
            if fv: xb = torch.flip(xb, (-2,))
            with torch.autocast(DEVICE, dtype=AMP):
                p = torch.sigmoid(MODEL(xb.contiguous(memory_format=torch.channels_last)).float())
            if fv: p = torch.flip(p, (-2,))
            if fh: p = torch.flip(p, (-1,))
            acc = p if acc is None else acc + p
    return (acc / 4)[:, 0].cpu().numpy()

def tstar_mask(p):
    """Mask maximising the expected IoU of a soft map: threshold J/(1+J) at the fixed point of J."""
    tot, J = float(p.sum()), 0.5
    if tot <= 0:
        return np.zeros(p.shape, bool), 0.0
    for _ in range(8):
        m = p >= J / (1 + J)
        inter = float(p[m].sum())
        Jn = inter / max(float(m.sum()) + tot - inter, 1e-9)
        if abs(Jn - J) < 1e-4:
            J = Jn
            break
        J = Jn
    return p >= J / (1 + J), J

def refine_image(name):
    """-> list over V4 candidates of (y0, x0, refined window mask) or None (keep V4's mask)."""
    n, wins, _ = CANDS[name]
    out, xs, geo = [None] * n, [], []
    for k in range(n):
        if wins[k] is None:
            continue
        M = frame_matrix(*frame_of(*seed_pixels(wins[k]))); box = frame_box(M)
        xs.append(make_input(name, wins[k], np.array([k + 1]), M, box)); geo.append((k, M, box))
    if not xs:
        return out
    probs = np.concatenate([predict(np.stack(xs[i:i + 128])) for i in range(0, len(xs), 128)])
    for (k, M, box), p in zip(geo, probs):
        pw = to_window(p, M, box)
        m, J = tstar_mask(pw)
        lab_, nl = ndi.label(m, structure=STRUCT8)
        if nl > 1:                                            # keep the piece that overlaps the seed most
            sw = seed_window(wins[k], box) > 0.5
            ov = np.bincount(lab_[sw], minlength=nl + 1)[1:]
            m = lab_ == (int(ov.argmax()) + 1 if ov.max() > 0 else int(np.bincount(lab_.ravel())[1:].argmax()) + 1)
        a0 = float(wins[k][2].sum())
        if 16 <= m.sum() and a0 / 3 <= m.sum() <= 3 * a0:      # guard: implausible redraws keep V4's mask
            y0, x0, y1, x1 = box
            ya, xa, yb, xb = max(y0, 0), max(x0, 0), min(y1, H), min(x1, W)
            out[k] = (ya, xa, m[ya - y0:yb - y0, xa - x0:xb - x0].copy())
    return out

REFINED = {}
for name in tqdm(PILOT, desc="Refine pilot fold"):
    REFINED[name] = refine_image(name)
save_pickle(RUN / f"refined_fold{CFG.pilot_fold}.pkl", REFINED)
push()
print("Refined:", sum(r is not None for v in REFINED.values() for r in v), "of", sum(len(v) for v in REFINED.values()), "candidates")


## Cell 15 — Verdict: same keep decisions (fusion run, cross-fitted v4_full), V4 masks vs refined masks


In [ ]:
FUS = fetch_fusion()
Z = np.load(FUS / "image_stats.npz", allow_pickle=False)
ZN = list(Z["names"]); OFF = Z["off"]
UKEY = [k for k in Z.files if k.startswith("v4_full__u")][0]
TAU = float([k for k in Z.files if k.startswith("v4_full__tau")][0].split("tau")[1])
U = Z[UKEY]
print("Keep decisions:", FUS.name, UKEY, "tau", TAU)

def paint(name, use_refined):
    """Kept candidates painted in descending score; a pixel belongs to the first candidate that claims it."""
    i = ZN.index(name)
    u = U[OFF[i]:OFF[i + 1]]
    n, wins, _ = CANDS[name]
    assert len(u) == n, f"{name}: {n} candidates here vs {len(u)} in the fusion run"
    lab = np.load(LAB_DIR / f"{Path(name).stem}.npy")
    out = np.zeros((H, W), np.int32); m = 0
    for k in np.argsort(-u):
        if u[k] < TAU:
            break
        r = REFINED[name][k] if use_refined else None
        if r is None:
            y0, x0, mk = wins[k]
        else:
            y0, x0, mk = r
        reg = out[y0:y0 + mk.shape[0], x0:x0 + mk.shape[1]]
        free = mk & (reg == 0)
        if free.sum() < 16:
            continue
        m += 1
        reg[free] = m
    return out, m

def image_stats(name, use_refined):
    lab, n = paint(name, use_refined)
    area = np.bincount(lab.ravel(), minlength=n + 1)[1:].astype(np.float64)
    st = np.zeros(5)
    for _, iou in overlap_matrices(lab, n, area, GT_INDEX[name]):
        st += match_stats(iou)
    best = [iou.max(1) if iou.shape[1] else np.zeros(iou.shape[0]) for _, iou in overlap_matrices(lab, n, area, GT_INDEX[name])]
    return st, np.concatenate(best) if best else np.zeros(0)

S4, S5, B4, B5 = [], [], [], []
for name in tqdm(PILOT, desc="Score"):
    a, ba = image_stats(name, False); b, bb = image_stats(name, True)
    S4.append(a); S5.append(b); B4.append(ba); B5.append(bb)
S4, S5, B4, B5 = np.array(S4), np.array(S5), np.concatenate(B4), np.concatenate(B5)
dates = np.array([Path(n).stem[:8] for n in PILOT])
from sklearn.model_selection import GroupKFold
splits = [te for _, te in GroupKFold(4).split(PILOT, groups=dates)]
pq4, pq5 = summarize(S4.sum(0)), summarize(S5.sum(0))
wins = int(sum(summarize(S5[te].sum(0))["pq"] > summarize(S4[te].sum(0))["pq"] for te in splits))
rng = np.random.default_rng(CFG.seed); ud = np.unique(dates); boots = []
for _ in range(2000):
    pick = np.concatenate([np.flatnonzero(dates == d) for d in rng.choice(ud, len(ud))])
    boots.append(summarize(S5[pick].sum(0))["pq"] - summarize(S4[pick].sum(0))["pq"])
near = (B4 >= 0.1) & (B4 <= 0.5)
VERDICT = {"pq_v4_masks": pq4["pq"], "pq_refined": pq5["pq"], "delta": pq5["pq"] - pq4["pq"],
           "delta_ci90": [float(np.percentile(boots, 5)), float(np.percentile(boots, 95))], "split_wins": f"{wins}/4",
           "sq": [pq4["sq"], pq5["sq"]], "rq": [pq4["rq"], pq5["rq"]], "tp_fp_fn_v4": [pq4["tp"], pq4["fp"], pq4["fn"]],
           "tp_fp_fn_refined": [pq5["tp"], pq5["fp"], pq5["fn"]],
           "near_misses_v4": int(near.sum()), "near_miss_converted": float((B5[near] > 0.5).mean()) if near.any() else 0.0,
           "matched_lost": int(((B4 > 0.5) & (B5 <= 0.5)).sum()),
           "gate": bool(pq5["pq"] - pq4["pq"] >= 0.003 and wins >= 3), "keep_decisions": FUS.name, "tau": TAU,
           "minutes": (time.time() - T0) / 60 if "T0" in globals() else None}
write_json(RUN / f"verdict_fold{CFG.pilot_fold}.json", VERDICT)
push()
print(json.dumps(VERDICT, indent=1))
